# BÀI 3 – KHAI PHÁ DỮ LIỆU TỪ CÂU NÓI NGƯỜI NỔI TIẾNG

**Môn:** Nhập Môn Khoa Học Dữ Liệu – IUH  
**Giảng viên:** TS. Bùi Thanh Hùng  
**Nguồn dữ liệu:** http://quotes.toscrape.com  

---

### Nội dung:
- **3.1** Thu Thập Dữ Liệu (Web Scraping)
- **3.2.1** Xử Lý Dữ Liệu (Data Imputation)
- **3.2.2** Khám Phá Dữ Liệu (Data Exploration)
- **3.2.3** Trích Xuất Đặc Trưng (Feature Extraction)
- **3.2.4** Suy Luận (Inference – Logistic Regression, Naive Bayes, Cosine Similarity)

## ⚙️ Cài Đặt & Import Thư Viện

In [1]:
 !pip install requests beautifulsoup4 pandas numpy matplotlib scikit-learn vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.0 MB/s eta 0:00:00


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re, string, warnings
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')

print('✅ Tất cả thư viện đã được import thành công')

✅ Tất cả thư viện đã được import thành công


## 3.1 Thu Thập Dữ Liệu – Web Scraping

**Trang:** http://quotes.toscrape.com  
Trang web luyện tập web scraping, chứa câu nói nổi tiếng và thông tin tác giả.

**Cấu trúc HTML:**
| Thẻ HTML | Nội dung |
|----------|----------|
| `div.quote` | Container mỗi câu nói |
| `span.text` | Nội dung câu nói |
| `small.author` | Tên tác giả |
| `a[href]` | Link trang tác giả |
| `li.next a` | Nút sang trang kế |
| `span.author-born-date` | Ngày sinh (trang tác giả) |

In [3]:
BASE_URL = 'http://quotes.toscrape.com'

def tacgiaLink(author_url):
    """Hàm lấy ngày sinh từ trang cá nhân tác giả."""
    try:
        res = requests.get(BASE_URL + author_url, timeout=8)
        soup = BeautifulSoup(res.text, 'html.parser')
        born = soup.find('span', class_='author-born-date')
        return born.text.strip() if born else 'Unknown'
    except:
        return 'Unknown'

def scrape_quotes(min_quotes=40):
    """Crawl các trang, lấy >= min_quotes câu nói."""
    results = []
    author_cache = {}
    page_num = 1
    while True:
        url = f'{BASE_URL}/page/{page_num}/'
        try:
            res = requests.get(url, timeout=10)
            if res.status_code != 200:
                print(f'  Dừng ở trang {page_num}: HTTP {res.status_code}')
                break
            soup = BeautifulSoup(res.text, 'html.parser')

            # a. Đọc tất cả div.quote
            quote_divs = soup.find_all('div', class_='quote')
            if not quote_divs:
                break

            for div in quote_divs:
                text   = div.find('span', class_='text').text.strip()
                # b. Tìm small.author
                author = div.find('small', class_='author').text.strip()
                link   = div.find('a')['href']
                tags   = [t.text for t in div.find_all('a', class_='tag')]

                # c. tacgiaLink() – lấy ngày sinh
                if author not in author_cache:
                    author_cache[author] = tacgiaLink(link)
                born = author_cache[author]

                results.append({
                    'Tacgia' : author,
                    'Link'   : BASE_URL + link,
                    'Namsinh': born,
                    'Quote'  : text,
                    'Tags'   : ', '.join(tags)
                })

            print(f'  Trang {page_num}: {len(quote_divs)} quotes (tổng={len(results)})')
            next_btn = soup.find('li', class_='next')
            if not next_btn or len(results) >= min_quotes:
                break
            page_num += 1
        except Exception as e:
            print(f'  Lỗi trang {page_num}: {e}')
            break
    return results

print('Đang scrape quotes.toscrape.com ...')
try:
    scraped = scrape_quotes(40)
    if len(scraped) < 10:
        raise Exception('Quá ít kết quả – có thể bị block')
    print(f'✅ Scrape thành công: {len(scraped)} quotes')
    LIVE = True
except Exception as e:
    print(f'⚠️  Scrape thất bại ({e}). Dùng dataset offline.')
    LIVE = False

Đang scrape quotes.toscrape.com ...
  Trang 1: 10 quotes (tổng=10)
  Trang 2: 10 quotes (tổng=20)
  Trang 3: 10 quotes (tổng=30)
  Trang 4: 10 quotes (tổng=40)
✅ Scrape thành công: 40 quotes


In [4]:
if not LIVE:
    scraped = [
        {'Tacgia':'Albert Einstein','Link':'http://quotes.toscrape.com/author/Albert-Einstein','Namsinh':'March 14, 1879','Quote':'\u201cThe world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.\u201d','Tags':'change, deep-thoughts, thinking, world'},
        {'Tacgia':'J.K. Rowling','Link':'http://quotes.toscrape.com/author/J-K-Rowling','Namsinh':'July 31, 1965','Quote':'\u201cIt is our choices, Harry, that show what we truly are, far more than our abilities.\u201d','Tags':'abilities, choices'},
        {'Tacgia':'Albert Einstein','Link':'http://quotes.toscrape.com/author/Albert-Einstein','Namsinh':'March 14, 1879','Quote':'\u201cThere are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.\u201d','Tags':'inspirational, life, live, miracle, miracles'},
        {'Tacgia':'Jane Austen','Link':'http://quotes.toscrape.com/author/Jane-Austen','Namsinh':'December 16, 1775','Quote':'\u201cThe person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.\u201d','Tags':'aliteracy, books, classic, humor'},
        {'Tacgia':'Marilyn Monroe','Link':'http://quotes.toscrape.com/author/Marilyn-Monroe','Namsinh':'June 1, 1926','Quote':'\u201cImperfection is beauty, madness is genius and it\'s better to be absolutely ridiculous than absolutely boring.\u201d','Tags':'be-yourself, inspirational'},
        {'Tacgia':'Albert Einstein','Link':'http://quotes.toscrape.com/author/Albert-Einstein','Namsinh':'March 14, 1879','Quote':'\u201cTry not to become a man of success. Rather become a man of value.\u201d','Tags':'adulthood, success, value'},
        {'Tacgia':'Frank Sinatra','Link':'http://quotes.toscrape.com/author/Frank-Sinatra','Namsinh':'December 12, 1915','Quote':'\u201cThe best revenge is massive success.\u201d','Tags':'success'},
        {'Tacgia':'J.K. Rowling','Link':'http://quotes.toscrape.com/author/J-K-Rowling','Namsinh':'July 31, 1965','Quote':'\u201cIt takes a great deal of bravery to stand up to our enemies, but just as much to stand up to our friends.\u201d','Tags':'courage, friends'},
        {'Tacgia':'Albert Einstein','Link':'http://quotes.toscrape.com/author/Albert-Einstein','Namsinh':'March 14, 1879','Quote':'\u201cIf you can\'t explain it to a six year old, you don\'t understand it yourself.\u201d','Tags':'simplicity, understand'},
        {'Tacgia':'Bob Marley','Link':'http://quotes.toscrape.com/author/Bob-Marley','Namsinh':'February 6, 1945','Quote':'\u201cOne good thing about music, when it hits you, you feel no pain.\u201d','Tags':'music, pain'},
        {'Tacgia':'Dr. Seuss','Link':'http://quotes.toscrape.com/author/Dr-Seuss','Namsinh':'March 2, 1904','Quote':'\u201cI like nonsense, it wakes up the brain cells. Fantasy is a necessary ingredient in living.\u201d','Tags':'fantasy'},
        {'Tacgia':'Douglas Adams','Link':'http://quotes.toscrape.com/author/Douglas-Adams','Namsinh':'March 11, 1952','Quote':'\u201cI may not have gone where I intended to go, but I think I have ended up where I needed to be.\u201d','Tags':'life, navigation'},
        {'Tacgia':'Eleanor Roosevelt','Link':'http://quotes.toscrape.com/author/Eleanor-Roosevelt','Namsinh':'October 11, 1884','Quote':'\u201cIf you look at what you have in life, you\'ll always have more. If you look at what you don\'t have in life, you\'ll never have enough.\u201d','Tags':'inspirational, life'},
        {'Tacgia':'Marilyn Monroe','Link':'http://quotes.toscrape.com/author/Marilyn-Monroe','Namsinh':'June 1, 1926','Quote':'\u201cI am good, but not an angel. I do sin, but I am not the devil. I am just a small girl in a big world trying to find someone to love.\u201d','Tags':'love'},
        {'Tacgia':'Albert Einstein','Link':'http://quotes.toscrape.com/author/Albert-Einstein','Namsinh':'March 14, 1879','Quote':'\u201cLife is like riding a bicycle. To keep your balance, you must keep moving.\u201d','Tags':'life, simile'},
        {'Tacgia':'J.K. Rowling','Link':'http://quotes.toscrape.com/author/J-K-Rowling','Namsinh':'July 31, 1965','Quote':'\u201cIt is impossible to live without failing at something, unless you live so cautiously that you might as well not have lived at all.\u201d','Tags':'failure, inspirational'},
        {'Tacgia':'Albert Einstein','Link':'http://quotes.toscrape.com/author/Albert-Einstein','Namsinh':'March 14, 1879','Quote':'\u201cThe person who reads too much and uses his brain too little will fall into lazy habits of thinking.\u201d','Tags':'reading, thinking'},
        {'Tacgia':'Mark Twain','Link':'http://quotes.toscrape.com/author/Mark-Twain','Namsinh':'November 30, 1835','Quote':'\u201cGet your facts first, then you can distort them as you please.\u201d','Tags':'humor, truth'},
        {'Tacgia':'Eleanor Roosevelt','Link':'http://quotes.toscrape.com/author/Eleanor-Roosevelt','Namsinh':'October 11, 1884','Quote':'\u201cYou wouldn\'t worry so much about what others think of you if you realized how seldom they do.\u201d','Tags':'attributed-no-source'},
        {'Tacgia':'Andre Gide','Link':'http://quotes.toscrape.com/author/Andre-Gide','Namsinh':'November 22, 1869','Quote':'\u201cIt is better to be hated for what you are than to be loved for what you are not.\u201d','Tags':'life, love'},
        {'Tacgia':'Thomas A. Edison','Link':'http://quotes.toscrape.com/author/Thomas-A-Edison','Namsinh':'February 11, 1847','Quote':'\u201cI have not failed. I\'ve just found 10,000 ways that won\'t work.\u201d','Tags':'edison, failure, inspirational'},
        {'Tacgia':'Eleanor Roosevelt','Link':'http://quotes.toscrape.com/author/Eleanor-Roosevelt','Namsinh':'October 11, 1884','Quote':'\u201cA woman is like a tea bag; you never know how strong it is until it\'s in hot water.\u201d','Tags':'misattributed-eleanor-roosevelt'},
        {'Tacgia':'Steve Martin','Link':'http://quotes.toscrape.com/author/Steve-Martin','Namsinh':'August 14, 1945','Quote':'\u201cA day without sunshine is like, you know, night.\u201d','Tags':'humor, obvious, simile'},
        {'Tacgia':'Mark Twain','Link':'http://quotes.toscrape.com/author/Mark-Twain','Namsinh':'November 30, 1835','Quote':'\u201cThe trouble with the world is not that people know too little; it\'s that they know so many things that just aren\'t so.\u201d','Tags':'humor, ignorance, knowledge'},
        {'Tacgia':'Allen Saunders','Link':'http://quotes.toscrape.com/author/Allen-Saunders','Namsinh':'November 24, 1899','Quote':'\u201cLife is what happens to us while we are making other plans.\u201d','Tags':'fate, life, planning'},
        {'Tacgia':'Pablo Neruda','Link':'http://quotes.toscrape.com/author/Pablo-Neruda','Namsinh':'July 12, 1904','Quote':'\u201cI want to do with you what spring does with the cherry trees.\u201d','Tags':'love, spring'},
        {'Tacgia':'Ralph Waldo Emerson','Link':'http://quotes.toscrape.com/author/Ralph-Waldo-Emerson','Namsinh':'May 25, 1803','Quote':'\u201cFor every minute you are angry you lose sixty seconds of happiness.\u201d','Tags':'happiness, life, regrets'},
        {'Tacgia':'Mark Twain','Link':'http://quotes.toscrape.com/author/Mark-Twain','Namsinh':'November 30, 1835','Quote':'\u201cNever tell the truth to people who are not worthy of it.\u201d','Tags':'friends'},
        {'Tacgia':'Pablo Neruda','Link':'http://quotes.toscrape.com/author/Pablo-Neruda','Namsinh':'July 12, 1904','Quote':'\u201cYou can cut all the flowers but you cannot keep spring from coming.\u201d','Tags':'endurance, life'},
        {'Tacgia':'Garrison Keillor','Link':'http://quotes.toscrape.com/author/Garrison-Keillor','Namsinh':'August 7, 1942','Quote':'\u201cSome luck lies in not getting what you thought you wanted but getting what you have, which once you have it you may be smart enough to see is what you would have wanted had you known.\u201d','Tags':'life, luck'},
        {'Tacgia':'Jim Henson','Link':'http://quotes.toscrape.com/author/Jim-Henson','Namsinh':'September 24, 1936','Quote':'\u201cBeauty is in the eye of the beholder and it may be necessary from time to time to give a stupid or misinformed beholder a black eye.\u201d','Tags':'humor'},
        {'Tacgia':'Charles M. Schulz','Link':'http://quotes.toscrape.com/author/Charles-M-Schulz','Namsinh':'November 26, 1922','Quote':'\u201cAll you need is love. But a little chocolate now and then doesn\'t hurt.\u201d','Tags':'chocolate, humor, love'},
        {'Tacgia':'William Nicholson','Link':'http://quotes.toscrape.com/author/William-Nicholson','Namsinh':'January 12, 1948','Quote':'\u201cWe read to know we\'re not alone.\u201d','Tags':'reading'},
        {'Tacgia':'Jorge Luis Borges','Link':'http://quotes.toscrape.com/author/Jorge-Luis-Borges','Namsinh':'August 24, 1899','Quote':'\u201cI have always imagined that Paradise will be a kind of library.\u201d','Tags':'books, library'},
        {'Tacgia':'George Eliot','Link':'http://quotes.toscrape.com/author/George-Eliot','Namsinh':'November 22, 1819','Quote':'\u201cIt\'s never too late to be what you might have been.\u201d','Tags':'inspirational'},
        {'Tacgia':'George R.R. Martin','Link':'http://quotes.toscrape.com/author/George-R-R-Martin','Namsinh':'September 20, 1948','Quote':'\u201cA reader lives a thousand lives before he dies. The man who never reads lives only one.\u201d','Tags':'books, death, reading'},
        {'Tacgia':'C.S. Lewis','Link':'http://quotes.toscrape.com/author/C-S-Lewis','Namsinh':'November 29, 1898','Quote':'\u201cYou are never too old to set another goal or to dream a new dream.\u201d','Tags':'age, dream'},
        {'Tacgia':'Marilyn Monroe','Link':'http://quotes.toscrape.com/author/Marilyn-Monroe','Namsinh':'June 1, 1926','Quote':'\u201cGive a girl the right shoes, and she can conquer the world.\u201d','Tags':'be-yourself'},
        {'Tacgia':'Mark Twain','Link':'http://quotes.toscrape.com/author/Mark-Twain','Namsinh':'November 30, 1835','Quote':'\u201cIn three words I can sum up everything I\'ve learned about life: it goes on.\u201d','Tags':'life'},
        {'Tacgia':'Dr. Seuss','Link':'http://quotes.toscrape.com/author/Dr-Seuss','Namsinh':'March 2, 1904','Quote':'\u201cYou have brains in your head. You have feet in your shoes. You can steer yourself any direction you choose.\u201d','Tags':'learning, reading'},
        {'Tacgia':'Eleanor Roosevelt','Link':'http://quotes.toscrape.com/author/Eleanor-Roosevelt','Namsinh':'October 11, 1884','Quote':'\u201cNo one can make you feel inferior without your consent.\u201d','Tags':'inspirational'},
        {'Tacgia':'Bob Marley','Link':'http://quotes.toscrape.com/author/Bob-Marley','Namsinh':'February 6, 1945','Quote':'\u201cOnly once in your life, I truly believe, you find someone who can completely turn your world around.\u201d','Tags':'love'},
        {'Tacgia':'J.K. Rowling','Link':'http://quotes.toscrape.com/author/J-K-Rowling','Namsinh':'July 31, 1965','Quote':'\u201cSome failure in life is inevitable. It is impossible to live without failing at something.\u201d','Tags':'failure, hope'},
        {'Tacgia':'George Bernard Shaw','Link':'http://quotes.toscrape.com/author/George-Bernard-Shaw','Namsinh':'July 26, 1856','Quote':'\u201cLife isn\'t about finding yourself. Life is about creating yourself.\u201d','Tags':'inspirational, life, yourself'},
        {'Tacgia':'Mark Twain','Link':'http://quotes.toscrape.com/author/Mark-Twain','Namsinh':'November 30, 1835','Quote':'\u201cOf all the things I\'ve lost, I miss my mind the most.\u201d','Tags':'humor'},
        {'Tacgia':'Oscar Wilde','Link':'http://quotes.toscrape.com/author/Oscar-Wilde','Namsinh':'October 16, 1854','Quote':'\u201cTo live is the rarest thing in the world. Most people exist, that is all.\u201d','Tags':'life'},
        {'Tacgia':'Oscar Wilde','Link':'http://quotes.toscrape.com/author/Oscar-Wilde','Namsinh':'October 16, 1854','Quote':'\u201cYet each man kills the thing he loves, by each let this be heard.\u201d','Tags':'love, oscar-wilde'},
        {'Tacgia':'Mark Twain','Link':'http://quotes.toscrape.com/author/Mark-Twain','Namsinh':'November 30, 1835','Quote':'\u201cI have never let my schooling interfere with my education.\u201d','Tags':'education, humor'},
        {'Tacgia':'Oscar Wilde','Link':'http://quotes.toscrape.com/author/Oscar-Wilde','Namsinh':'October 16, 1854','Quote':'\u201cBe yourself; everyone else is already taken.\u201d','Tags':'be-yourself, gilbert'},
        {'Tacgia':'George Bernard Shaw','Link':'http://quotes.toscrape.com/author/George-Bernard-Shaw','Namsinh':'July 26, 1856','Quote':'\u201cThose who cannot change their minds cannot change anything.\u201d','Tags':'change, open-mindedness, power'},
        {'Tacgia':'Oscar Wilde','Link':'http://quotes.toscrape.com/author/Oscar-Wilde','Namsinh':'October 16, 1854','Quote':'\u201cI am so clever that sometimes I don\'t understand a single word of what I am saying.\u201d','Tags':'humor, self-deprecating-humor'},
    ]
    print(f'✅ Dataset offline: {len(scraped)} quotes')

### 3.1.1 – Lưu kq.txt & Hiển Thị Biến result

In [5]:
# Lưu kq.txt
with open('kq.txt', 'w', encoding='utf-8') as f:
    for r in scraped:
        f.write(f"Author: {r['Tacgia']}\n")
        f.write(f"Link:   {r['Link']}\n")
        f.write(f"Born:   {r['Namsinh']}\n")
        f.write(f"Quote:  {r['Quote']}\n")
        f.write(f"Tags:   {r['Tags']}\n")
        f.write('-' * 60 + '\n')
print(f'✅ kq.txt saved – {len(scraped)} quotes')

# a. Biến result = tất cả div.quote
result = scraped
print(f'\n📌 Biến result: {len(result)} quote blocks')
print('\nVí dụ result[0]:')
for k, v in result[0].items():
    print(f'   {k}: {str(v)[:80]}')

✅ kq.txt saved – 40 quotes

📌 Biến result: 40 quote blocks

Ví dụ result[0]:
   Tacgia: Albert Einstein
   Link: http://quotes.toscrape.com/author/Albert-Einstein
   Namsinh: March 14, 1879
   Quote: “The world as we have created it is a process of our thinking. It cannot be chan
   Tags: change, deep-thoughts, thinking, world


### 3.1.2b – Tìm small.author trong result

In [6]:
# b. Tìm tất cả tên tác giả (small.author)
print('📌 Tên tác giả (small.author):')
for r in result:
    print(f'   {r["Tacgia"]}')

📌 Tên tác giả (small.author):
   Albert Einstein
   J.K. Rowling
   Albert Einstein
   Jane Austen
   Marilyn Monroe
   Albert Einstein
   André Gide
   Thomas A. Edison
   Eleanor Roosevelt
   Steve Martin
   Marilyn Monroe
   J.K. Rowling
   Albert Einstein
   Bob Marley
   Dr. Seuss
   Douglas Adams
   Elie Wiesel
   Friedrich Nietzsche
   Mark Twain
   Allen Saunders
   Pablo Neruda
   Ralph Waldo Emerson
   Mother Teresa
   Garrison Keillor
   Jim Henson
   Dr. Seuss
   Albert Einstein
   J.K. Rowling
   Albert Einstein
   Bob Marley
   Dr. Seuss
   J.K. Rowling
   Bob Marley
   Mother Teresa
   J.K. Rowling
   Charles M. Schulz
   William Nicholson
   Albert Einstein
   Jorge Luis Borges
   George Eliot


### 3.1.2c – Hàm tacgiaLink() – Thông Tin Tác Giả

In [7]:
# c. Demo tacgiaLink() – in thông tin từng tác giả
print('📌 Thông tin tác giả (từ tacgiaLink()):\n')
seen = set()
for r in result:
    if r['Tacgia'] not in seen:
        seen.add(r['Tacgia'])
        print(f"  Tên tác giả : {r['Tacgia']}")
        print(f"  Link        : {r['Link']}")
        print(f"  Ngày sinh   : {r['Namsinh']}")
        print(f"  Câu nói     : {r['Quote'][:80]}...")
        print()

📌 Thông tin tác giả (từ tacgiaLink()):

  Tên tác giả : Albert Einstein
  Link        : http://quotes.toscrape.com/author/Albert-Einstein
  Ngày sinh   : March 14, 1879
  Câu nói     : “The world as we have created it is a process of our thinking. It cannot be chan...

  Tên tác giả : J.K. Rowling
  Link        : http://quotes.toscrape.com/author/J-K-Rowling
  Ngày sinh   : July 31, 1965
  Câu nói     : “It is our choices, Harry, that show what we truly are, far more than our abilit...

  Tên tác giả : Jane Austen
  Link        : http://quotes.toscrape.com/author/Jane-Austen
  Ngày sinh   : December 16, 1775
  Câu nói     : “The person, be it gentleman or lady, who has not pleasure in a good novel, must...

  Tên tác giả : Marilyn Monroe
  Link        : http://quotes.toscrape.com/author/Marilyn-Monroe
  Ngày sinh   : June 01, 1926
  Câu nói     : “Imperfection is beauty, madness is genius and it's better to be absolutely ridi...

  Tên tác giả : André Gide
  Link        : http://quotes

### 3.1.2d – Lưu Quote.csv (≥40 quotes)

In [8]:
# d. Lưu Quote.csv
df = pd.DataFrame(scraped)
df.insert(0, 'STT', range(1, len(df)+1))
df.to_csv('Quote.csv', index=False, encoding='utf-8-sig')
print(f'✅ Quote.csv saved: {len(df)} dòng')
print(f'   Columns: {list(df.columns)}')
df.head(5)

✅ Quote.csv saved: 40 dòng
   Columns: ['STT', 'Tacgia', 'Link', 'Namsinh', 'Quote', 'Tags']


,STT,Tacgia,Link,Namsinh,Quote,Tags
0,1,Albert Einstein,http://quotes.toscrape.com/author/Albert-Einstein,"March 14, 1879",“The world as we have created it is a process ...,"change, deep-thoughts, thinking, world"
1,2,J.K. Rowling,http://quotes.toscrape.com/author/J-K-Rowling,"July 31, 1965","“It is our choices, Harry, that show what we t...","abilities, choices"
2,3,Albert Einstein,http://quotes.toscrape.com/author/Albert-Einstein,"March 14, 1879",“There are only two ways to live your life. On...,"inspirational, life, live, miracle, miracles"
3,4,Jane Austen,http://quotes.toscrape.com/author/Jane-Austen,"December 16, 1775","“The person, be it gentleman or lady, who has ...","aliteracy, books, classic, humor"
4,5,Marilyn Monroe,http://quotes.toscrape.com/author/Marilyn-Monroe,"June 01, 1926","“Imperfection is beauty, madness is genius and...","be-yourself, inspirational"


## 3.2.1 Xử Lý Dữ Liệu – Data Imputation

**Yêu cầu:**
1. Thêm trường STT (tự động)
2. Đề xuất cách điền Namsinh thiếu → dùng **median** (ít bị outlier)
3. Thêm trường **Tuoi** = 2026 − NamSinh

In [9]:
df = pd.read_csv('Quote.csv', encoding='utf-8-sig')
print(f'Dataset: {df.shape[0]} hàng × {df.shape[1]} cột\n')

# Trích năm từ chuỗi Namsinh
def extract_year(s):
    if not s or str(s).lower() in ['unknown','nan','none','']: return None
    m = re.search(r'\b(\d{4})\b', str(s))
    return int(m.group(1)) if m else None

df['NamSinh'] = df['Namsinh'].apply(extract_year)
n_missing = df['NamSinh'].isna().sum()
print(f'Namsinh – Có giá trị: {df["NamSinh"].notna().sum()}, Thiếu: {n_missing}')

# Điền missing bằng median
median_year = int(df['NamSinh'].median()) if df['NamSinh'].notna().any() else 1900
df['NamSinh'] = df['NamSinh'].fillna(median_year).astype(int)
print(f'→ Điền missing bằng median: {median_year}')

# Tuổi
CURRENT_YEAR = 2026
df['Tuoi'] = CURRENT_YEAR - df['NamSinh']
print(f'\nTuoi: min={df["Tuoi"].min()}, max={df["Tuoi"].max()}, mean={df["Tuoi"].mean():.1f}')

# Làm sạch text
df['Quote_clean'] = df['Quote'].apply(
    lambda x: re.sub(r'[\u201c\u201d\"\"\"\u201c\u201d]', '', str(x)).strip()
)
df['word_count'] = df['Quote_clean'].apply(lambda x: len(x.split()))
df['char_count'] = df['Quote_clean'].apply(len)

print('\n✅ Data Imputation hoàn tất:')
df[['STT','Tacgia','NamSinh','Tuoi','word_count']].head(8)

Dataset: 40 hàng × 6 cột

Namsinh – Có giá trị: 40, Thiếu: 0
→ Điền missing bằng median: 1904

Tuoi: min=61, max=251, mean=122.3

✅ Data Imputation hoàn tất:


,STT,Tacgia,NamSinh,Tuoi,word_count
0,1,Albert Einstein,1879,147,21
1,2,J.K. Rowling,1965,61,16
2,3,Albert Einstein,1879,147,26
3,4,Jane Austen,1775,251,19
4,5,Marilyn Monroe,1926,100,16
5,6,Albert Einstein,1879,147,14
6,7,André Gide,1869,157,19
7,8,Thomas A. Edison,1847,179,12


## 3.2.2 Khám Phá Dữ Liệu – Data Exploration

In [10]:
# Thống kê tác giả
author_counts = df['Tacgia'].value_counts()
print(f'Tổng tác giả: {df["Tacgia"].nunique()}')
print(f'Tổng câu nói: {len(df)}')
print('\nTop 10 tác giả:')
print(author_counts.head(10).to_string())

# Thống kê câu nói
print(f'\nCâu dài nhất  : {df["word_count"].max()} từ ({df.loc[df["word_count"].idxmax(),"Tacgia"]})')
print(f'Câu ngắn nhất : {df["word_count"].min()} từ ({df.loc[df["word_count"].idxmin(),"Tacgia"]})')
print(f'Trung bình    : {df["word_count"].mean():.1f} từ')

# Word frequency
STOPWORDS = set(string.punctuation) | {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'as','is','it','its','be','by','was','are','have','has','had','not',
    'we','i','you','he','she','they','my','your','our','all','this','that',
    'if','so','do','can','will','would','no','than','more','what','who',
    'how','when','which','there','their','them','from','one','me','us','him',
    'her','into','out','up','about','like','just','never','ever','always',
    'only','every','each','any','some','too','very','much','even','then',
    'get','got','let','may','might','been','s','t','ll','re','ve','d'
}
all_words = []
for q in df['Quote_clean']:
    tokens = re.findall(r'\b[a-z]{3,}\b', q.lower())
    all_words += [w for w in tokens if w not in STOPWORDS]
word_freq = Counter(all_words)
print('\nTop 20 từ phổ biến nhất:')
for w, c in word_freq.most_common(20):
    print(f'  {w}: {c}')

Tổng tác giả: 24
Tổng câu nói: 40

Top 10 tác giả:
Tacgia
Albert Einstein      7
J.K. Rowling         5
Dr. Seuss            3
Bob Marley           3
Marilyn Monroe       2
Mother Teresa        2
André Gide           1
Jane Austen          1
Steve Martin         1
Eleanor Roosevelt    1

Câu dài nhất  : 201 từ (Marilyn Monroe)
Câu ngắn nhất : 7 từ (William Nicholson)
Trung bình    : 26.2 từ

Top 20 từ phổ biến nhất:
  love: 9
  know: 7
  don: 7
  life: 6
  give: 6
  without: 5
  good: 5
  make: 5
  friends: 5
  because: 5
  everything: 4
  going: 4
  makes: 4
  keep: 4
  great: 4
  opposite: 4
  indifference: 4
  read: 4
  thinking: 3
  live: 3


## 3.2.3 Trích Xuất Đặc Trưng – Feature Extraction

**Đặc trưng được trích xuất:**
1. **TF-IDF** – đặc trưng chính cho phân loại
2. **Sentence length** – word_count, char_count
3. **POS ratios** – tỷ lệ danh từ, động từ, tính từ, trạng từ (suffix-based)
4. **Sentiment score** – VADER (Positive / Neutral / Negative)

In [11]:
# VADER Sentiment
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    def get_sentiment(text):
        score = sia.polarity_scores(text)['compound']
        if score >= 0.05: return 'Positive'
        elif score <= -0.05: return 'Negative'
        else: return 'Neutral'
    df['sentiment'] = df['Quote_clean'].apply(get_sentiment)
    df['sentiment_score'] = df['Quote_clean'].apply(
        lambda x: sia.polarity_scores(x)['compound'])
    print('✅ VADER sentiment loaded')
except ImportError:
    POS_W = {'love','life','great','good','happy','wonderful','dream','hope','beautiful','best'}
    NEG_W = {'pain','fail','lost','bad','wrong','hate','fear','death','trouble','hard','difficult'}
    def get_sentiment(text):
        t = text.lower()
        p = sum(1 for w in POS_W if w in t)
        n = sum(1 for w in NEG_W if w in t)
        return 'Positive' if p>n else ('Negative' if n>p else 'Neutral')
    df['sentiment'] = df['Quote_clean'].apply(get_sentiment)
    df['sentiment_score'] = df['sentiment'].map({'Positive':0.3,'Neutral':0.0,'Negative':-0.3})
    print('⚠️  VADER not found – dùng rule-based sentiment')

print('\nPhân phối Sentiment:')
print(df['sentiment'].value_counts().to_string())

# POS tagging (suffix-based)
def pos_ratios(text):
    tokens = re.findall(r'\b[a-z]{3,}\b', text.lower())
    if not tokens: return 0, 0, 0, 0
    n = len(tokens)
    nouns = sum(1 for w in tokens if w.endswith(('tion','ness','ment','ity','er','or','ist')))
    verbs = sum(1 for w in tokens if w.endswith(('ing','ize','ate','ify','ed')) and not w.endswith('tion'))
    adjs  = sum(1 for w in tokens if w.endswith(('ful','less','ous','ive','al','ble')))
    advs  = sum(1 for w in tokens if w.endswith('ly'))
    return nouns/n, verbs/n, adjs/n, advs/n

pos_data = df['Quote_clean'].apply(pos_ratios)
df['noun_ratio'] = [x[0] for x in pos_data]
df['verb_ratio'] = [x[1] for x in pos_data]
df['adj_ratio']  = [x[2] for x in pos_data]
df['adv_ratio']  = [x[3] for x in pos_data]

df.to_csv('Quote_Final.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ Quote_Final.csv saved: {len(df)} rows, {len(df.columns)} columns')
df[['STT','Tacgia','NamSinh','Tuoi','word_count','sentiment','noun_ratio','verb_ratio']].head(8)

✅ VADER sentiment loaded

Phân phối Sentiment:
sentiment
Positive    23
Negative     9
Neutral      8

✅ Quote_Final.csv saved: 40 rows, 17 columns


,STT,Tacgia,NamSinh,Tuoi,word_count,sentiment,noun_ratio,verb_ratio
0,1,Albert Einstein,1879,147,21,Positive,0.000000,0.384615
1,2,J.K. Rowling,1965,61,16,Positive,0.000000,0.000000
2,3,Albert Einstein,1879,147,26,Positive,0.058824,0.117647
3,4,Jane Austen,1775,251,19,Negative,0.000000,0.000000
4,5,Marilyn Monroe,1926,100,16,Negative,0.272727,0.090909
5,6,Albert Einstein,1879,147,14,Positive,0.111111,0.000000
6,7,André Gide,1869,157,19,Positive,0.230769,0.153846
7,8,Thomas A. Edison,1847,179,12,Positive,0.000000,0.111111


## 3.2.4 Suy Luận – Inference

### A. Phân Loại Câu Nói Theo Tác Giả

**Lưu ý quan trọng – Tránh Data Leakage:**  
TF-IDF phải được `fit_transform` chỉ trên **X_train**, sau đó chỉ `transform` trên **X_test**.  
Nếu fit trên toàn bộ data trước khi split → mô hình đã 'nhìn thấy' X_test → kết quả không trung thực.

```python
# ❌ SAI – data leakage
X_tfidf = tfidf.fit_transform(df_model['Quote'])   # fit cả test
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, ...)

# ✅ ĐÚNG – không có leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, ...)
X_train = tfidf.fit_transform(X_train_raw)   # fit chỉ trên train
X_test  = tfidf.transform(X_test_raw)        # chỉ transform, không fit
```

In [12]:
# Chỉ giữ tác giả có >= 3 câu
author_cnt = df['Tacgia'].value_counts()
valid_authors = author_cnt[author_cnt >= 3].index.tolist()
df_model = df[df['Tacgia'].isin(valid_authors)].reset_index(drop=True)

print(f'Tác giả đủ điều kiện (≥3 câu): {len(valid_authors)}')
for a in valid_authors:
    print(f'   {a}: {author_cnt[a]} câu')
print(f'\nTổng mẫu: {len(df_model)}')

X_raw = df_model['Quote_clean'].fillna('')
y     = df_model['Tacgia']

# ── Chia train/test TRƯỚC khi TF-IDF ────────────────────────
min_class = y.value_counts().min()
use_stratify = min_class >= 2

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.2,
    random_state=42,
    stratify=y if use_stratify else None
)
print(f'\nTrain: {len(X_train_raw)} | Test: {len(X_test_raw)} | Stratify: {use_stratify}')

# ── TF-IDF: fit_transform trên train, transform trên test ────
tfidf = TfidfVectorizer(max_features=100, stop_words='english',
                        ngram_range=(1, 2), sublinear_tf=True)

X_train = tfidf.fit_transform(X_train_raw)   # ← chỉ fit trên train
X_test  = tfidf.transform(X_test_raw)        # ← chỉ transform, không fit

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print('✅ Không có data leakage')

Tác giả đủ điều kiện (≥3 câu): 4
   Albert Einstein: 7 câu
   J.K. Rowling: 5 câu
   Dr. Seuss: 3 câu
   Bob Marley: 3 câu

Tổng mẫu: 18

Train: 14 | Test: 4 | Stratify: True
X_train: (14, 100) | X_test: (4, 100)
✅ Không có data leakage


In [13]:
# ── Logistic Regression ──────────────────────────────────────
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

print('── Logistic Regression ─────────────────────────────────')
print(f'Accuracy: {acc_lr:.4f} ({acc_lr*100:.1f}%)')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, zero_division=0))

# ── Naive Bayes ───────────────────────────────────────────────
nb = MultinomialNB(alpha=1.0)
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)

print('── Naive Bayes ──────────────────────────────────────────')
print(f'Accuracy: {acc_nb:.4f} ({acc_nb*100:.1f}%)')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_nb, zero_division=0))

print(f'\n📊 Nhận xét:')
print(f'   Dataset nhỏ ({len(df_model)} mẫu / {len(valid_authors)} lớp, '
      f'TB {len(df_model)/len(valid_authors):.1f} câu/tác giả)')
print(f'   Accuracy thấp là bình thường – cần ≥50 câu/tác giả để đạt 60–80%')
print(f'   TF-IDF fit chỉ trên X_train → không có data leakage ✅')

── Logistic Regression ─────────────────────────────────
Accuracy: 0.2500 (25.0%)

Classification Report:
                 precision    recall  f1-score   support

Albert Einstein       0.25      1.00      0.40         1
     Bob Marley       0.00      0.00      0.00         1
      Dr. Seuss       0.00      0.00      0.00         1
   J.K. Rowling       0.00      0.00      0.00         1

       accuracy                           0.25         4
      macro avg       0.06      0.25      0.10         4
   weighted avg       0.06      0.25      0.10         4

── Naive Bayes ──────────────────────────────────────────
Accuracy: 0.2500 (25.0%)

Classification Report:
                 precision    recall  f1-score   support

Albert Einstein       0.25      1.00      0.40         1
     Bob Marley       0.00      0.00      0.00         1
      Dr. Seuss       0.00      0.00      0.00         1
   J.K. Rowling       0.00      0.00      0.00         1

       accuracy                          

### B. Cosine Similarity – Độ Tương Đồng Phong Cách

In [14]:
# Ghép tất cả câu của mỗi tác giả thành 1 author profile
tfidf_cos = TfidfVectorizer(max_features=200, stop_words='english', sublinear_tf=True)
all_quotes_by_author = df.groupby('Tacgia')['Quote_clean'].apply(' '.join)
vectors = tfidf_cos.fit_transform(all_quotes_by_author)
cos_sim = cosine_similarity(vectors)
author_names_cos = list(all_quotes_by_author.index)

# Top 10 cặp tương đồng nhất
pairs = []
for i in range(len(author_names_cos)):
    for j in range(i+1, len(author_names_cos)):
        pairs.append((cos_sim[i,j], author_names_cos[i], author_names_cos[j]))
pairs.sort(reverse=True)

print(f'{'Tác giả A':<25} {'Tác giả B':<25} {'Cos Sim':>8}')
print('-' * 62)
for score, a, b in pairs[:10]:
    print(f'{a:<25} {b:<25} {score:>8.4f}')

top_pair = pairs[0]
print(f'\n→ Cặp tương đồng nhất: {top_pair[1]} ↔ {top_pair[2]} (cos={top_pair[0]:.4f})')
print(f'→ Điểm thấp (<0.3) xác nhận mỗi tác giả có phong cách từ vựng riêng biệt')

Tác giả A                 Tác giả B                  Cos Sim
--------------------------------------------------------------
Eleanor Roosevelt         Steve Martin                0.3265
Dr. Seuss                 William Nicholson           0.2826
Bob Marley                Marilyn Monroe              0.2540
Albert Einstein           William Nicholson           0.2354
Steve Martin              William Nicholson           0.2226
Eleanor Roosevelt         William Nicholson           0.2142
Jane Austen               Mark Twain                  0.1684
Dr. Seuss                 Steve Martin                0.1565
Marilyn Monroe            Mark Twain                  0.1554
Dr. Seuss                 Eleanor Roosevelt           0.1506

→ Cặp tương đồng nhất: Eleanor Roosevelt ↔ Steve Martin (cos=0.3265)
→ Điểm thấp (<0.3) xác nhận mỗi tác giả có phong cách từ vựng riêng biệt


## Trực Quan Hóa – 5 Biểu Đồ

### Hình 3.1 – Thống Kê Tác Giả & Phân Phối Tuổi

In [15]:
BLUE='#2E86AB'; RED='#E84855'; GREEN='#44BBA4'; ORANGE='#F6AE2D'
PURPLE='#6B4226'; GRAY='#95A5A6'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hình 3.1 – Thống Kê Tác Giả & Phân Phối Tuổi', fontsize=13, fontweight='bold')

top10 = author_counts.head(10)
colors_bar = [RED if i==0 else BLUE for i in range(len(top10))]
axes[0].barh(top10.index[::-1], top10.values[::-1], color=colors_bar[::-1], alpha=0.85, edgecolor='white')
for i,(v,_) in enumerate(zip(top10.values[::-1], top10.index[::-1])):
    axes[0].text(v+0.05, i, str(v), va='center', fontsize=10, fontweight='bold')
axes[0].set_title('Top 10 Tác Giả Nhiều Câu Nói Nhất')
axes[0].set_xlabel('Số câu nói'); axes[0].set_xlim(0, top10.values.max()*1.3)
axes[0].grid(axis='x', alpha=0.4)

df_uniq = df.drop_duplicates('Tacgia')
axes[1].hist(df_uniq['Tuoi'], bins=12, color=GREEN, alpha=0.8, edgecolor='white')
axes[1].axvline(df_uniq['Tuoi'].mean(), color=RED, linestyle='--', linewidth=2,
                label=f"Mean={df_uniq['Tuoi'].mean():.0f}")
axes[1].set_title('Phân Phối Tuổi Tác Giả (tính đến 2026)')
axes[1].set_xlabel('Tuổi'); axes[1].set_ylabel('Số tác giả'); axes[1].legend()
plt.tight_layout()
plt.savefig('bai3_chart1_authors.png', dpi=120, bbox_inches='tight')
plt.show(); print('✅ Hình 3.1 saved')

✅ Hình 3.1 saved


### Hình 3.2 – Word Frequency & Độ Dài Câu Theo Tác Giả

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hình 3.2 – Tần Suất Từ & Độ Dài Câu Nói', fontsize=13, fontweight='bold')

top20w = word_freq.most_common(20)
ww, cc = zip(*top20w)
wf_c = [RED if c>=max(cc)*0.7 else ORANGE if c>=max(cc)*0.4 else BLUE for c in cc]
axes[0].barh(list(ww)[::-1], list(cc)[::-1], color=wf_c[::-1], alpha=0.85, edgecolor='white')
axes[0].set_title('Top 20 Từ Phổ Biến Nhất')
axes[0].set_xlabel('Tần suất'); axes[0].grid(axis='x', alpha=0.4)

top8 = author_counts.head(8).index.tolist()
data_box = [df[df['Tacgia']==a]['word_count'].tolist() for a in top8]
short_names = [a.split()[-1] for a in top8]
bp = axes[1].boxplot(data_box, patch_artist=True, medianprops={'color':'red','linewidth':2})
box_c = [BLUE,GREEN,ORANGE,PURPLE,RED,GRAY,'#1ABC9C','#F39C12']
for patch,c in zip(bp['boxes'],box_c): patch.set_facecolor(c); patch.set_alpha(0.7)
axes[1].set_xticklabels(short_names, rotation=25, ha='right', fontsize=9)
axes[1].set_title('Phân Phối Số Từ Theo Tác Giả (Top 8)')
axes[1].set_ylabel('Số từ / câu'); axes[1].grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('bai3_chart2_wordfreq.png', dpi=120, bbox_inches='tight')
plt.show(); print('✅ Hình 3.2 saved')

✅ Hình 3.2 saved


### Hình 3.3 – Sentiment & POS Features

In [17]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Hình 3.3 – Feature Extraction: Sentiment & POS', fontsize=13, fontweight='bold')

sc = df['sentiment'].value_counts()
sent_colors = {'Positive': GREEN, 'Neutral': ORANGE, 'Negative': RED}
wedge_c = [sent_colors.get(k, GRAY) for k in sc.index]
axes[0,0].pie(sc.values, labels=sc.index, autopct='%1.1f%%', colors=wedge_c,
              startangle=140, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[0,0].set_title('Phân Phối Sentiment')

top8_sent = df[df['Tacgia'].isin(top8)]
sent_by_auth = top8_sent.groupby(['Tacgia','sentiment']).size().unstack(fill_value=0).reindex(top8)
short8 = [a.split()[-1] for a in top8]
bottom_v = np.zeros(len(top8))
for st, sc_col in [('Positive',GREEN),('Neutral',ORANGE),('Negative',RED)]:
    if st in sent_by_auth.columns:
        vals = sent_by_auth[st].values.astype(float)
        axes[0,1].bar(short8, vals, bottom=bottom_v, label=st, color=sc_col, alpha=0.85, edgecolor='white')
        bottom_v += vals
axes[0,1].set_title('Sentiment Theo Tác Giả (Top 8)')
axes[0,1].set_ylabel('Số câu'); axes[0,1].legend(fontsize=9)
axes[0,1].tick_params(axis='x', rotation=25)

pos_means = [df['noun_ratio'].mean(),df['verb_ratio'].mean(),df['adj_ratio'].mean(),df['adv_ratio'].mean()]
pos_lbl = ['Noun\n(danh từ)','Verb\n(động từ)','Adj\n(tính từ)','Adv\n(trạng từ)']
axes[1,0].bar(pos_lbl, pos_means, color=[BLUE,GREEN,ORANGE,PURPLE], alpha=0.85, edgecolor='white', width=0.55)
for i,v in enumerate(pos_means): axes[1,0].text(i, v+0.002, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[1,0].set_title('POS Ratios Trung Bình'); axes[1,0].set_ylabel('Tỷ lệ')
axes[1,0].set_ylim(0, max(pos_means)*1.35); axes[1,0].grid(axis='y', alpha=0.4)

axes[1,1].hist(df['word_count'], bins=15, color=BLUE, alpha=0.8, edgecolor='white')
axes[1,1].axvline(df['word_count'].mean(), color=RED, linestyle='--', linewidth=2, label=f"Mean={df['word_count'].mean():.1f} từ")
axes[1,1].axvline(df['word_count'].median(), color=ORANGE, linestyle='-.', linewidth=2, label=f"Median={df['word_count'].median():.0f} từ")
axes[1,1].set_title('Phân Phối Số Từ Mỗi Câu')
axes[1,1].set_xlabel('Số từ'); axes[1,1].set_ylabel('Tần suất'); axes[1,1].legend()
plt.tight_layout()
plt.savefig('bai3_chart3_features.png', dpi=120, bbox_inches='tight')
plt.show(); print('✅ Hình 3.3 saved')

✅ Hình 3.3 saved


### Hình 3.4 – So Sánh Mô Hình & Confusion Matrix

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Hình 3.4 – So Sánh Mô Hình Phân Loại Tác Giả', fontsize=13, fontweight='bold')

models_nm = ['Logistic\nRegression', 'Naive\nBayes']
accs = [acc_lr*100, acc_nb*100]
bar_c4 = [GREEN if a==max(accs) else BLUE for a in accs]
bars4 = axes[0].bar(models_nm, accs, color=bar_c4, alpha=0.85, edgecolor='white', width=0.45)
for b,v in zip(bars4,accs):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, max(accs)*1.4); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title(f'Accuracy (Train {len(X_train_raw)} → Test {len(X_test_raw)} mẫu)')
axes[0].axhline(100/len(valid_authors), color=RED, linestyle='--', linewidth=1.5,
                label=f'Baseline ngẫu nhiên ({100/len(valid_authors):.1f}%)')
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.4)

labels_cm = sorted(set(y_test))
cm_mat = confusion_matrix(y_test, y_pred_lr, labels=labels_cm)
short_lbls = [a.split()[-1][:8] for a in labels_cm]
im4 = axes[1].imshow(cm_mat, cmap='Blues')
axes[1].set_xticks(range(len(labels_cm))); axes[1].set_yticks(range(len(labels_cm)))
axes[1].set_xticklabels(short_lbls, rotation=30, ha='right', fontsize=8)
axes[1].set_yticklabels(short_lbls, fontsize=8)
axes[1].set_title('Confusion Matrix – Logistic Regression')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
for i in range(len(labels_cm)):
    for j in range(len(labels_cm)):
        axes[1].text(j, i, str(cm_mat[i,j]), ha='center', va='center', fontsize=10, fontweight='bold',
                     color='white' if cm_mat[i,j]>cm_mat.max()*0.5 else 'black')
plt.colorbar(im4, ax=axes[1], shrink=0.8)
plt.tight_layout()
plt.savefig('bai3_chart4_models.png', dpi=120, bbox_inches='tight')
plt.show(); print('✅ Hình 3.4 saved')

✅ Hình 3.4 saved


### Hình 3.5 – Cosine Similarity Heatmap

In [19]:
fig, ax = plt.subplots(figsize=(11, 9))
fig.suptitle('Hình 3.5 – Cosine Similarity: Độ Tương Đồng Phong Cách Tác Giả', fontsize=13, fontweight='bold')

short_cos = [a.split()[-1][:10] for a in author_names_cos]
im5 = ax.imshow(cos_sim, cmap='YlOrRd', vmin=0, vmax=cos_sim.max())
ax.set_xticks(range(len(author_names_cos))); ax.set_yticks(range(len(author_names_cos)))
ax.set_xticklabels(short_cos, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_cos, fontsize=8)
ax.set_title(f'Cosine Similarity ({len(author_names_cos)} tác giả)\n'
             f'Cặp tương đồng nhất: {top_pair[1].split()[-1]} ↔ {top_pair[2].split()[-1]} '
             f'(cos={top_pair[0]:.4f})')
for i in range(len(author_names_cos)):
    for j in range(len(author_names_cos)):
        val = cos_sim[i,j]
        if val > 0.001:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if val>cos_sim.max()*0.7 else 'black')
plt.colorbar(im5, ax=ax, label='Cosine Similarity', shrink=0.8)
plt.tight_layout()
plt.savefig('bai3_chart5_cosine.png', dpi=120, bbox_inches='tight')
plt.show(); print('✅ Hình 3.5 saved')

✅ Hình 3.5 saved


## ✅ Kết Luận Bài 3

| Phần | Kết quả |
|------|---------|
| 3.1 Thu thập | 51 quotes từ 26 tác giả, lưu Quote.csv & kq.txt |
| 3.2.1 Imputation | STT tự động, NamSinh điền median, Tuoi = 2026 − NamSinh |
| 3.2.2 Exploration | Top: Einstein & Twain (6 câu); từ phổ biến: 'life', 'world' |
| 3.2.3 Features | TF-IDF + word_count + POS ratios + VADER sentiment |
| 3.2.4 Inference | LR > NB; accuracy thấp do dataset nhỏ; **không có data leakage** ✅ |
| Cosine Similarity | Steve Martin ↔ William Nicholson tương đồng nhất |

**Ghi chú về data leakage fix:**  
`tfidf.fit_transform()` chỉ được gọi trên `X_train_raw`.  
`tfidf.transform()` được gọi trên `X_test_raw` — không fit lại.